# RF Fixed Split vs RF Walk-Forward vs Ridge Walk-Forward

Three-way comparison:
- **RF fixed split** — train on first 80%, evaluate on last 20%
- **RF walk-forward** — expanding window from main pipeline (`all_results`)
- **Ridge walk-forward** — same walk-forward scheme with Ridge regression

**Purpose:** understand how much of the performance difference is due to
the evaluation scheme (fixed vs walk-forward) vs the model (RF vs Ridge).

**Requires:** `all_results` from the main pipeline in memory.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────────
# Fraction of waves to use as hold-out (last N% chronologically)
HOLDOUT_FRAC = 0.20

results_fixed = {}

for series_name, res in all_results.items():
    Z         = res["Z"]
    y         = res["y"]
    waves_df  = res["waves_df"]
    bp        = res["best_params"]
    n         = len(y)

    n_holdout = max(1, int(round(HOLDOUT_FRAC * n)))
    n_train   = n - n_holdout

    Z_tr, Z_ho = Z[:n_train],  Z[n_train:]
    y_tr, y_ho = y[:n_train],  y[n_train:]
    w_tr,  w_ho = waves_df.iloc[:n_train], waves_df.iloc[n_train:]

    # Sample weights
    sw = np.log1p(res["waves_df"]["n_after_filter"].values[:n_train])
    sw = sw / sw.mean()

    rf = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=bp["min_samples_leaf"],
        max_features=bp["max_features"],
        random_state=SEED, n_jobs=-1,
    )
    rf.fit(Z_tr, y_tr, sample_weight=sw)

    y_pred_tr = rf.predict(Z_tr)
    y_pred_ho = rf.predict(Z_ho)

    # Metrics
    r2_in  = 1 - np.sum((y_tr - y_pred_tr)**2) / np.sum((y_tr - y_tr.mean())**2)
    rmse_ho = np.sqrt(mean_squared_error(y_ho, y_pred_ho))
    y_ar1   = np.concatenate([[y_tr[-1]], y_ho[:-1]])
    ar1_rmse = np.sqrt(mean_squared_error(y_ho, y_ar1))
    r2_ho   = 1 - np.sum((y_ho - y_pred_ho)**2) / np.sum((y_ho - y_ho.mean())**2)
    dy_true = np.diff(y_ho)
    dy_pred = np.diff(y_pred_ho)
    dir_acc = np.mean(np.sign(dy_true) == np.sign(dy_pred)) if len(dy_true) > 0 else np.nan

    results_fixed[series_name] = {
        "r2_insample": r2_in,
        "rmse_holdout": rmse_ho,
        "ar1_rmse": ar1_rmse,
        "r2_holdout": r2_ho,
        "dir_acc": dir_acc,
        "n_train": n_train,
        "n_holdout": n_holdout,
        "y_tr": y_tr, "y_pred_tr": y_pred_tr,
        "y_ho": y_ho, "y_pred_ho": y_pred_ho,
        "w_tr": w_tr, "w_ho": w_ho,
    }

    print(f"\n{series_name}  (train={n_train}, holdout={n_holdout})")
    print(f"  In-sample R²:     {r2_in:.3f}")
    print(f"  Hold-out R²:      {r2_ho:.3f}")
    print(f"  Hold-out RMSE:    {rmse_ho:.4f}  |  AR(1): {ar1_rmse:.4f}")
    print(f"  Directional acc:  {dir_acc:.1%}")

In [ ]:
# ── Ridge walk-forward ────────────────────────────────────────────────────────
def walk_forward_ridge(Z, y, min_train, alphas=None):
    """Walk-forward Ridge with GCV alpha selection on initial window."""
    if alphas is None:
        alphas = np.logspace(-3, 6, 40)
    # Tune alpha once on initial window
    sc    = StandardScaler().fit(Z[:min_train])
    Zs    = sc.transform(Z)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rc = RidgeCV(alphas=alphas, gcv_mode="auto",
                     scoring="neg_mean_squared_error")
        rc.fit(Zs[:min_train], y[:min_train])
    best_alpha = rc.alpha_

    # Walk-forward loop at fixed alpha
    records = []
    for t in range(min_train, len(y)):
        sc_t  = StandardScaler().fit(Z[:t])
        Z_tr  = sc_t.transform(Z[:t])
        Z_va  = sc_t.transform(Z[t:t+1])
        m     = Ridge(alpha=best_alpha).fit(Z_tr, y[:t])
        records.append({
            "realized":  float(y[t]),
            "predicted": float(m.predict(Z_va)[0]),
        })

    wf    = pd.DataFrame(records)
    wf["error"]        = wf["realized"] - wf["predicted"]
    wf["sq_error"]     = wf["error"] ** 2
    wf["ar1_pred"]     = np.concatenate([[y[min_train-1]], wf["realized"].values[:-1]])
    wf["ar1_sq_error"] = (wf["realized"] - wf["ar1_pred"]) ** 2

    rmse     = np.sqrt(wf["sq_error"].mean())
    ar1_rmse = np.sqrt(wf["ar1_sq_error"].mean())
    y_wf     = wf["realized"].values
    r2       = 1 - wf["sq_error"].sum() / np.sum((y_wf - y_wf.mean())**2)
    dy_true  = np.diff(y_wf)
    dy_pred  = np.diff(wf["predicted"].values)
    dir_acc  = np.mean(np.sign(dy_true) == np.sign(dy_pred)) if len(dy_true) > 0 else np.nan

    return {"rmse": rmse, "ar1_rmse": ar1_rmse, "r2": r2,
            "dir_acc": dir_acc, "wf_df": wf, "alpha": best_alpha}


results_ridge_wf = {}
for series_name, res in all_results.items():
    Z         = res["Z"]
    y         = res["y"]
    min_train = res["min_train"]
    print(f"{series_name}  (min_train={min_train}, eval steps={len(y)-min_train})")
    r = walk_forward_ridge(Z, y, min_train)
    results_ridge_wf[series_name] = r
    print(f"  Ridge α={r['alpha']:.2e}  RMSE={r['rmse']:.4f}  "
          f"AR(1)={r['ar1_rmse']:.4f}  R²={r['r2']:.3f}  DirAcc={r['dir_acc']:.1%}")

In [ ]:
# ── Three-way comparison table ────────────────────────────────────────────────
print("\n── Model Comparison ──────────────────────────────────────────────────────────")
print(f"{'Series':<22} {'Metric':<16} {'RF Fixed':>10} {'RF WalkFwd':>12} {'Ridge WalkFwd':>14}")
print("─" * 78)

for series_name in all_results:
    wf_rf    = all_results[series_name]
    fix      = results_fixed[series_name]
    wf_ridge = results_ridge_wf[series_name]

    metrics = [
        ("Hold-out R²",   fix["r2_holdout"],   wf_rf["wf_r2"],   wf_ridge["r2"]),
        ("RMSE",          fix["rmse_holdout"],  wf_rf["wf_rmse"], wf_ridge["rmse"]),
        ("AR(1) RMSE",    fix["ar1_rmse"],      wf_rf["ar1_rmse"],wf_ridge["ar1_rmse"]),
        ("Dir. accuracy", fix["dir_acc"],        wf_rf["dir_acc"], wf_ridge["dir_acc"]),
    ]
    for i, (metric, fval, rfval, ridgeval) in enumerate(metrics):
        name_col = series_name if i == 0 else ""
        if metric == "Dir. accuracy":
            fs    = f"{fval:.1%}"
            rfs   = f"{rfval:.1%}"
            ridgs = f"{ridgeval:.1%}"
        else:
            fs    = f"{fval:.4f}"
            rfs   = f"{rfval:.4f}"
            ridgs = f"{ridgeval:.4f}"
        print(f"{name_col:<22} {metric:<16} {fs:>10} {rfs:>12} {ridgs:>14}")
    print()

In [ ]:
n_series = len(all_results)
fig, axes = plt.subplots(n_series, 3, figsize=(17, 5 * n_series))
if n_series == 1:
    axes = axes[None, :]
fig.suptitle("RF Fixed Split vs RF Walk-Forward vs Ridge Walk-Forward",
             fontweight="bold", fontsize=12)

for row, series_name in enumerate(all_results):
    wf_rf    = all_results[series_name]
    fix      = results_fixed[series_name]
    wf_ridge = results_ridge_wf[series_name]
    waves_df = wf_rf["waves_df"]
    min_train= wf_rf["min_train"]
    y        = wf_rf["y"]
    wf_dates = waves_df["wave_date"].values[min_train:]

    # Panel 1: RF fixed split scatter
    ax = axes[row, 0]
    ax.scatter(fix["y_tr"], fix["y_pred_tr"], alpha=0.35, color="C0",
               s=18, label=f"Train (R²={fix['r2_insample']:.3f})")
    ax.scatter(fix["y_ho"], fix["y_pred_ho"], alpha=0.8,  color="C3",
               s=25, label=f"Hold-out (R²={fix['r2_holdout']:.3f})")
    lo = min(fix["y_tr"].min(), fix["y_pred_tr"].min())
    hi = max(fix["y_tr"].max(), fix["y_pred_tr"].max())
    ax.plot([lo,hi],[lo,hi],"k--",lw=1)
    ax.set_xlabel("Realized"); ax.set_ylabel("Predicted")
    ax.set_title(f"{series_name.replace('_',' ').title()}\nRF — fixed split")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 2: RF walk-forward time series
    ax = axes[row, 1]
    wf_df_rf = wf_rf["wf_df"]
    ax.plot(waves_df["wave_date"].values[:min_train], y[:min_train],
            "-", color="gray", alpha=0.3, lw=1, label="Tuning window")
    ax.plot(wf_dates, wf_df_rf["realized"],  "-",  color="C0", lw=1.5,
            label="Realized")
    ax.plot(wf_dates, wf_df_rf["predicted"], "--", color="C0", lw=1.5,
            label=f"RF WF (R²={wf_rf['wf_r2']:.3f})")
    ax.axvline(waves_df["wave_date"].values[min_train],
               color="gray", ls=":", alpha=0.7)
    ax.set_title("RF — walk-forward")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

    # Panel 3: Ridge walk-forward time series
    ax = axes[row, 2]
    wf_df_ridge = wf_ridge["wf_df"]
    ax.plot(waves_df["wave_date"].values[:min_train], y[:min_train],
            "-", color="gray", alpha=0.3, lw=1, label="Tuning window")
    ax.plot(wf_dates, wf_df_ridge["realized"],  "-",  color="C0", lw=1.5,
            label="Realized")
    ax.plot(wf_dates, wf_df_ridge["predicted"], "--", color="C2", lw=1.5,
            label=f"Ridge WF (R²={wf_ridge['r2']:.3f})")
    ax.axvline(waves_df["wave_date"].values[min_train],
               color="gray", ls=":", alpha=0.7)
    ax.set_title(f"Ridge — walk-forward  (α={wf_ridge['alpha']:.1e})")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("rf_vs_ridge_walkforward.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved rf_vs_ridge_walkforward.png")

In [ ]:
# ── Overfit diagnostic ────────────────────────────────────────────────────────
print("\n── Overfit diagnostic ────────────────────────────────────────────────────")
print(f"{'Series':<25} {'In-sample R²':>14} {'Hold-out R²':>13} {'Gap':>8}")
print("─" * 65)
for series_name, fix in results_fixed.items():
    gap  = fix["r2_insample"] - fix["r2_holdout"]
    flag = "  ⚠ large gap" if gap > 0.5 else ""
    print(f"{series_name:<25} {fix['r2_insample']:>14.3f} "
          f"{fix['r2_holdout']:>13.3f} {gap:>8.3f}{flag}")

print()
print("A large in-sample vs hold-out R² gap indicates overfitting.")
print("Walk-forward R² is the more honest metric — it uses every")
print("available observation as an out-of-sample test point.")